## 0. 설치 — 완료 후 세션 재시작


In [1]:
!nvidia-smi

Sat Oct 10 17:23:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
%pip install "vllm==0.11.2"
%pip install "huggingface_hub>=0.34,<2"
%pip install "pyngrok>=7,<9"
%pip install "requests>=2.32,<3"
%pip install "bitsandbytes>=0.46.1,<0.49"

## 1. GPU와 설정 확인


In [1]:
import os
import sys
import subprocess
import json
import time
from pathlib import Path
import torch
from importlib.metadata import version

if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임과 드라이버를 확인하세요.")
print(torch.cuda.get_device_name(0))

if torch.cuda.get_device_properties(0).total_memory / 2**30 < 14:
    raise RuntimeError("이 설정은 T4 16GB급 이상의 GPU를 대상으로 합니다.")
print({name: version(name) for name in ("vllm", "torch", "bitsandbytes")})

if version("vllm") != "0.11.2":
    raise RuntimeError("설치 셀 실행 후 세션을 재시작하세요. 이 노트북은 vLLM 0.11.2 설정입니다.")

free, total = torch.cuda.mem_get_info()
print(f"GPU free/total: {free / 2**30:.2f}/{total / 2**30:.2f} GiB")

WORK = Path("/content/jurisflow-serving")
WORK.mkdir(parents=True, exist_ok=True)
TP_SIZE = 1
ADAPTER_ID = "grant88/qwen3-8b-jurisflow-qlora-mixed-adapter"
ADAPTER_REVISION = "main"
BASE_MODEL_OVERRIDE = None
TOKENIZER_OVERRIDE = None
MODEL_NAME = "jurisflow"
PORT = 8001
MAX_MODEL_LEN = 2048
GPU_MEMORY_UTILIZATION = 0.85
print(f"사용 GPU 수: {TP_SIZE}, 컨텍스트: {MAX_MODEL_LEN}")


Tesla T4
{'vllm': '0.11.2', 'torch': '2.9.0', 'bitsandbytes': '0.48.2'}
GPU free/total: 14.46/14.56 GiB
사용 GPU 수: 1, 컨텍스트: 2048


## 2. Hugging Face 인증과 LoRA 설정 읽기


In [3]:
from google.colab import userdata
from huggingface_hub import snapshot_download, hf_hub_download


HF_TOKEN = userdata.get('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError("저장소 접근 권한이 있는 HF 토큰을 입력하세요.")

adapter_path = Path(snapshot_download(
    ADAPTER_ID, revision=ADAPTER_REVISION, token=HF_TOKEN,
    allow_patterns=["adapter_config.json", "adapter_model.safetensors", "adapter_model.bin"],
))

config = json.loads((adapter_path / "adapter_config.json").read_text())

if config.get("peft_type") != "LORA":
    raise ValueError("이 노트북은 PEFT LoRA 어댑터용입니다.")

if not any((adapter_path / n).is_file() for n in ("adapter_model.safetensors", "adapter_model.bin")):
    raise ValueError("어댑터 가중치 파일이 없습니다.")

BASE_MODEL = BASE_MODEL_OVERRIDE or config.get("base_model_name_or_path")
if not BASE_MODEL:
    raise ValueError("BASE_MODEL_OVERRIDE에 학습에 사용한 베이스 모델을 지정하세요.")

rank = max([config["r"], *config.get("rank_pattern", {}).values()])
MAX_LORA_RANK = next((r for r in (8, 16, 32, 64, 128, 256, 320, 512) if r >= rank), None)

if MAX_LORA_RANK is None:
    raise ValueError(f"지원 범위를 넘는 LoRA rank: {rank}")

print({"base_model": BASE_MODEL, "adapter_rank": rank, "max_lora_rank": MAX_LORA_RANK})


base_config_path = hf_hub_download(BASE_MODEL, "config.json", token=HF_TOKEN)
base_config = json.loads(Path(base_config_path).read_text())
quant_config = base_config.get("quantization_config") or {}
print("Base quantization:", quant_config)
if quant_config.get("quant_method") != "bitsandbytes" or not quant_config.get("load_in_4bit"):
    raise ValueError("T4 구성에는 BitsAndBytes 4비트 베이스 모델이 필요합니다. 학습 베이스 모델 설정을 확인하세요.")


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/175M [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

{'base_model': 'unsloth/Qwen3-8B-unsloth-bnb-4bit', 'adapter_rank': 16, 'max_lora_rank': 16}


config.json: 0.00B [00:00, ?B/s]

Base quantization: {'_load_in_4bit': True, '_load_in_8bit': False, 'bnb_4bit_compute_dtype': 'bfloat16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'model.layers.34.self_attn', 'model.layers.34.mlp', 'model.layers.0.mlp', 'model.layers.1.mlp', 'model.layers.2.mlp', 'model.layers.4.mlp', 'model.layers.6.mlp'], 'llm_int8_threshold': 6.0, 'load_in_4bit': True, 'load_in_8bit': False, 'quant_method': 'bitsandbytes'}


## 3. vLLM 내장 서버 시작

In [5]:
import socket
import signal
from importlib.metadata import version

if globals().get("server_process") is not None and server_process.poll() is None:
    raise RuntimeError("서버가 이미 실행 중입니다. 종료 셀부터 실행하세요.")

if "llm" in globals():
    raise RuntimeError("기존 모델이 남아 있습니다. 세션을 재시작하세요.")

with socket.socket() as sock:
    if sock.connect_ex(("127.0.0.1", PORT)) == 0:
        raise RuntimeError(f"{PORT} 포트가 사용 중입니다.")

API_KEY = userdata.get('API_KEY')

if len(API_KEY) < 32:
    raise ValueError("32자 이상 키를 입력하세요.")

command = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", BASE_MODEL, "--tokenizer", TOKENIZER_OVERRIDE or BASE_MODEL,
    "--host", "127.0.0.1", "--port", str(PORT),
    "--quantization", "bitsandbytes", "--load-format", "bitsandbytes",
    "--dtype", "float16", "--tensor-parallel-size", "1",
    "--enable-lora", "--max-lora-rank", str(MAX_LORA_RANK),
    "--lora-modules", json.dumps({"name": MODEL_NAME,
        "path": str(adapter_path), "base_model_name": BASE_MODEL}),
    "--max-model-len", str(MAX_MODEL_LEN), "--max-num-seqs", "1",
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--enforce-eager", "--generation-config", "vllm", "--disable-log-requests",
]

server_env = os.environ.copy()
server_env.pop("VLLM_ENABLE_V1_MULTIPROCESSING", None)
server_env.update(HF_TOKEN=HF_TOKEN, VLLM_API_KEY=API_KEY,
                  CUDA_VISIBLE_DEVICES="0", VLLM_WORKER_MULTIPROC_METHOD="spawn",
                  VLLM_LOGGING_LEVEL="INFO", PYTHONUNBUFFERED="1")
LOG_PATH = WORK / "vllm.log"

with LOG_PATH.open("w") as log_file:
    server_process = subprocess.Popen(command, env=server_env,
        stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True)

del server_env
print("Server PID:", server_process.pid)
print("로그:", LOG_PATH)


Server PID: 10488
로그: /content/jurisflow-serving/vllm.log


## 4. 시작 대기와 오류 확인

In [6]:
import requests

local = requests.Session()
local.trust_env = False
LOCAL_URL = f"http://127.0.0.1:{PORT}"
HEADERS = {"Authorization": f"Bearer {API_KEY}"}

def tail_log(chars=10000):
    text = LOG_PATH.read_text(errors="replace")[-chars:]
    for secret in (API_KEY, HF_TOKEN):
        if secret:
            text = text.replace(secret, "[REDACTED]")
    print(text)

deadline = time.monotonic() + 1800

while time.monotonic() < deadline:
    if server_process.poll() is not None:
        tail_log()
        raise RuntimeError("서버 시작 실패. 위 로그의 첫 구체적인 예외를 확인하세요.")
    try:
        response = local.get(LOCAL_URL + "/v1/models", headers=HEADERS, timeout=5)
        if response.ok and MODEL_NAME in [m["id"] for m in response.json()["data"]]:
            print("서버 준비 완료:", LOCAL_URL)
            break
    except requests.RequestException:
        pass
    print("모델 다운로드/로딩 중…", flush=True)
    time.sleep(15)
else:
    tail_log()
    raise TimeoutError("대기 시간 초과. 서버는 남아 있으므로 종료 셀로 정리하세요.")


모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
모델 다운로드/로딩 중…
서버 준비 완료: http://127.0.0.1:8001


In [7]:
PAYLOAD = {
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "안녕하세요. 한국어로 짧게 자기소개해 주세요."}],
    "max_tokens": 128, "temperature": 0.6,
    "chat_template_kwargs": {"enable_thinking": False},
}

def check_api(session, base_url, headers):
    unauth = session.get(f"{base_url}/v1/models", headers={"ngrok-skip-browser-warning": "1"}, timeout=30)

    if unauth.status_code not in (401, 403):
        raise RuntimeError(f"인증 거부 검사 실패: HTTP {unauth.status_code}")

    models = session.get(f"{base_url}/v1/models", headers=headers, timeout=30)
    models.raise_for_status()

    if MODEL_NAME not in [m['id'] for m in models.json()['data']]:
        raise RuntimeError("어댑터가 API 모델 목록에 없습니다.")

    response = session.post(f"{base_url}/v1/chat/completions", headers=headers, json=PAYLOAD, timeout=180)
    response.raise_for_status()
    content = response.json()['choices'][0]['message'].get('content')

    if not content or not content.strip():
        raise RuntimeError("답변이 비어 있습니다.")
    print(content)

check_api(local, LOCAL_URL, HEADERS)
print("로컬 인증·모델 등록·추론 검사 통과")


안녕하세요. 저는 대학원생으로, 현재 정보공학과에서 연구하고 있으며, 특히 인공지능 분야에 관심이 많습니다. 연구와 학문적 성장에 열심히 노력하고 있으며, 다양한 기술 도구와 방법론을 익히고 있습니다.
로컬 인증·모델 등록·추론 검사 통과


## 5. ngrok HTTPS 터널과 외부 경로 검사


In [10]:
from pyngrok import ngrok
from pyngrok.conf import PyngrokConfig

if globals().get("tunnel") is not None:
    raise RuntimeError("터널이 이미 있습니다. 정리 셀 실행 후 다시 시작하세요.")

NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')

if not NGROK_AUTHTOKEN:
    raise ValueError("ngrok authtoken이 필요합니다.")

ngrok_config = PyngrokConfig(auth_token=NGROK_AUTHTOKEN)
tunnel = ngrok.connect(addr=f"127.0.0.1:{PORT}", proto="http", bind_tls=True,
                       inspect=False, pyngrok_config=ngrok_config)
PUBLIC_URL = tunnel.public_url

if not PUBLIC_URL.startswith("https://"):
    ngrok.disconnect(PUBLIC_URL, pyngrok_config=ngrok_config)
    tunnel = None
    raise RuntimeError("HTTPS 터널이 아닙니다.")
try:
    with requests.Session() as public_session:
        check_api(public_session, PUBLIC_URL, {
            **HEADERS, "ngrok-skip-browser-warning": "1"})
except Exception:
    ngrok.disconnect(PUBLIC_URL, pyngrok_config=ngrok_config)
    tunnel = None
    raise

print("공개 HTTPS 경로의 인증·모델 등록·추론 검사 통과")
print("외부 API base URL:", PUBLIC_URL + "/v1")


안녕하세요. 저는 한국어로 짧게 자기소개해 주겠습니다. 저는 인공지능 어시스턴트로서 여러 가지 정보를 제공하고, 사용자의 질문에 답하는 것을 목표로 하고 있습니다.
공개 HTTPS 경로의 인증·모델 등록·추론 검사 통과
외부 API base URL: https://fame-heat-motivator.ngrok-free.dev/v1


In [11]:
message = "지식재산권이란 무엇인가요?"

response = requests.post(
    f"{PUBLIC_URL}/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "ngrok-skip-browser-warning": "1",
    },
    json={
        "model": "jurisflow",
        "messages": [
            {"role": "user", "content": message}
        ],
        "max_tokens": 512,
        "temperature": 0.6,
        "stream": False,
        "chat_template_kwargs": {
            "enable_thinking": False
        },
    },
    timeout=180,
)

if response.ok:
    result = response.json()
    print("\n답변:")
    print(result["choices"][0]["message"]["content"])
    print("\n종료 사유:", result["choices"][0]["finish_reason"])
else:
    print("HTTP 오류:", response.status_code)
    print(response.text)


답변:
지식재산권은 창작물의 창작자가 그 창작물을 보호받을 권리를 말합니다. 이는 저작권, 특허권, 디자인권 등으로 구분되며, 각각의 권리에는 독점적이고 법적으로 보호되는 특성을 갖습니다.

종료 사유: stop


## 6. 종료


In [13]:
# 실제 종료 시 True 변경
STOP_SERVER = True

if STOP_SERVER:
    if globals().get("tunnel") is not None:
        ngrok.disconnect(tunnel.public_url, pyngrok_config=ngrok_config)
        tunnel = None
    if globals().get("server_process") is not None:
        try:
            os.killpg(server_process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            server_process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(server_process.pid, signal.SIGKILL)
            server_process.wait(timeout=10)

    print("터널과 서버 종료 요청 완료. 잔여 GPU 프로세스는 nvidia-smi로 확인하세요.")


터널과 서버 종료 요청 완료. 잔여 GPU 프로세스는 nvidia-smi로 확인하세요.


In [16]:
!nvidia-smi

Sat Oct 10 18:18:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             34W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----